In [2]:
# Packages
import duckdb
import os
import pandas as pd
# Local functions
from mimic_pipeline.duck import *
from mimic_pipeline.icustays import *
from mimic_pipeline.io_registry import register_parquet, register_parquet_general

# Prep ddb connection for data exploration
ddb = duckdb.connect("data/data.duckdb")
ddb.execute(f"PRAGMA memory_limit='{"8GB"}';")
ddb.execute(f"PRAGMA temp_directory='data/.duckdb_tmp';")

base_path = "data/preprocessing_checkpoints/"
directory_map = {
    "FINAL_MEASUREMENT_TABLE" : "final_measurement_table.parquet",
    "LABELS_33DAY": "labels_33day.parquet",
}
filepath_map = {
    key: base_path + value for key, value in directory_map.items()
}
register_parquet_general(ddb, filepath_map)

base_path = "../models/"
directory_map = {
    "FINAL_TIMESERIES" : "final_timeseries.parquet",
}
filepath_map = {
    key: base_path + value for key, value in directory_map.items()
}
register_parquet_general(ddb, filepath_map)


Done: Loaded FINAL_MEASUREMENT_TABLE table in 7.92s
Done: Loaded LABELS_33DAY table in 0.01s
Done: Loaded FINAL_TIMESERIES table in 4.45s


In [3]:
peek(ddb, "FINAL_MEASUREMENT_TABLE")
count(ddb, "FINAL_MEASUREMENT_TABLE")

┌──────────┬────────────────┬───────────────┬──────────┬──────────┬─────────────────┬───────────────────┬────────────────┬──────────────────┬────────────┬──────────────┬──────────────┬────────────────┬───────────────────┬───────────────────┬─────────────────────┬───────────────────────┬──────────────────────┬────────────────────────┬────────────────┬──────────────────┬────────────────────┬──────────────────────┬─────────────────────┬───────────────────────┬───────────┬─────────────┬─────────────┬───────────────┬─────────────────────┬───────────────────────┬────────────┬──────────────┬──────────────┬────────────────┬────────────────────────────┬──────────────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────────┬──────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┬────────────────────┬──────────────────────┬──────────────────┬────────────────────┬────────────────┬────────────────

1079778

In [4]:
peek(ddb, "LABELS_33DAY")
count(ddb, "LABELS_33DAY")

┌──────────┬─────────────────────┬─────────────────────────┬─────────────────────┬────────────────┬────────────┐
│ stay_id  │ survival_time_hours │ original_event_time_bin │ original_event_flag │ event_time_bin │ event_flag │
│  int64   │       double        │          int32          │        int32        │     int32      │   int32    │
├──────────┼─────────────────────┼─────────────────────────┼─────────────────────┼────────────────┼────────────┤
│ 35381922 │                27.0 │                       6 │                   0 │              6 │          0 │
│ 35854653 │                18.0 │                       4 │                   0 │              4 │          0 │
│ 37982806 │                44.0 │                      11 │                   0 │             11 │          0 │
│ 39911537 │                22.0 │                       5 │                   0 │              5 │          0 │
│ 38984565 │                46.0 │                      11 │                   0 │             1

52021

In [5]:
ddb.query("SELECT COUNT (*) FROM LABELS_33DAY WHERE event_flag = 1;").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         5016 │
└──────────────┘



In [6]:
peek(ddb, "FINAL_TIMESERIES")
count(ddb, "FINAL_TIMESERIES")

┌──────────┬────────────────┬───────────────┬──────────┬──────────┬─────────────────┬───────────────────┬────────────────┬──────────────────┬───────────────────┬──────────────┬────────────────────┬────────────────┬─────────────────┬───────────────────┬─────────────────────┬───────────────────────┬──────────────────────┬────────────────────────┬───────────────────┬──────────────────┬────────────────────┬──────────────────────┬─────────────────────┬───────────────────────┬───────────────────┬─────────────┬─────────────┬───────────────┬─────────────────────┬───────────────────────┬────────────┬──────────────┬──────────────┬────────────────┬────────────────────────────┬──────────────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────────┬──────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┬────────────────────┬──────────────────────┬──────────────────┬────────────────────┬───────────

1079778